# ЛР-03: Социальная защита в зимний период

## Student notebook: civil 02

Этот notebook предназначен для самостоятельного анализа.

Готовых численных ответов и заполненного sensitivity-разбора здесь нет.

## 1. Зачем нужен этот кейс

Здесь важно понять, какой ресурс становится главным узким местом зимой.

После выполнения работы студент должен уметь:

1. отделять активные ограничения от тех, где остаётся запас;
2. интерпретировать тени цен без длинного ручного симплекс-анализа;
3. сравнивать прогноз по `shadow price` с фактом;
4. делать управленческий вывод по дефицитности ресурсов.

## 2. Исходные данные

### Ограничения ресурсов

| Ресурс | Лимит |
| --- | --- |
| Бюджет | 96 |
| Трудозатраты | 60 |
| Операционная ёмкость | 50 |

### Программы

| Программа | Эффект | Бюджет | Трудозатраты | Операционная ёмкость |
| --- | --- | --- | --- | --- |
| Пункты обогрева | 97 | 46 | 24 | 18 |
| Продуктовые сертификаты | 76 | 24 | 16 | 9 |
| Социальные патрули | 70 | 18 | 14 | 12 |
| Срочный ремонт жилья | 92 | 44 | 28 | 22 |

## 3. Как читать прямую и двойственную задачи

Прямая задача выбирает масштабы программ `x_j`: это обычный план действий. Двойственная задача смотрит на ту же модель как в зеркале: вместо выбора программ она назначает внутренние цены ограничениям.

- `y_i` относится к ресурсному ограничению и показывает shadow price ресурса.
- `z_j` относится к верхней границе `x_j <= 1` и показывает ценность разрешения расширить программу выше 100 процентов.
- `slack` показывает остаток ресурса, а `binding` показывает, что ресурс исчерпан.

Единица измерения теневой цены: `единицы эффекта / единица соответствующего ограничения`. Нулевая теневая цена означает только то, что дополнительная единица этого ресурса локально не улучшает текущий оптимум. Для больших изменений прогноз нужно проверять повторным решением, потому что shadow price является локальной оценкой.

## 4. Что нужно сделать

1. Запишите прямую модель через переменные масштабов программ.
2. Объясните, почему двойственная задача является зеркальным взглядом на ограничения прямой задачи.
3. Подготовьте `c`, `A_ub`, `b_ub`, `bounds` для `linprog`.
4. Определите активные ограничения и запас ресурса.
5. Кратко запишите двойственную модель с переменными `y_i` и `z_j`.
6. Укажите единицу измерения каждой теневой цены.
7. Проведите минимум два сценария по `b` и один сценарий по `c`.
8. Сравните локальный прогноз по теневой цене с фактическим пересчётом.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# Шаг 1. Задаем исходные данные прямой задачи.

effects = np.array(
    [
        97,
        76,
        70,
        92,
    ],
    dtype=float,
)

A_ub = np.array(
    [
        [46, 24, 18, 44],
        [24, 16, 14, 28],
        [18, 9, 12, 22],
    ],
    dtype=float,
)

b_ub = np.array(
    [
        96,
        60,
        50,
    ],
    dtype=float,
)

bounds = [(0, 1)] * len(effects)

# Шаг 2. Формируем нейтральные подписи для табличной проверки данных.
program_names = [f"Программа {idx + 1}" for idx in range(len(effects))]
resource_names = [f"Ресурс {idx + 1}" for idx in range(len(b_ub))]

effects_df = pd.DataFrame(
    {
        "программа": program_names,
        "эффект на единицу": effects,
    }
)
A_ub_df = pd.DataFrame(
    A_ub,
    index=resource_names,
    columns=program_names,
)
b_ub_df = pd.DataFrame(
    {
        "ресурс": resource_names,
        "лимит": b_ub,
    }
)

print("Число программ =", len(effects))
print("Число ресурсных ограничений =", len(b_ub))
print("Вектор эффектов:")
display(effects_df)

print("Матрица ресурсных коэффициентов A_ub:")
display(A_ub_df)

print("Вектор правых частей b_ub:")
display(b_ub_df)

## 5. Шаблон для самостоятельной сборки решения

Сначала решите прямую задачу, потом переходите к binding/slack, dual и sensitivity. На каждом шаге держите в голове зеркало: `x_j` описывает действия, а `y_i` и `z_j` объясняют ценность ограничений.

In [ ]:
# TODO: реализуйте helper-функции после ручной записи прямой и двойственной моделей.
def solve_primal(effects, A_ub, b_ub, bounds):
    """Решает прямую задачу максимизации через `linprog`.

    Аргументы:
        effects (np.ndarray): Вектор эффектов программ.
        A_ub (np.ndarray): Матрица расхода ресурсов по программам.
        b_ub (np.ndarray): Вектор доступных лимитов ресурсов.
        bounds (list[tuple[float, float]]): Границы переменных `0 <= x <= 1`.

    Возвращает:
        tuple: Пара `result, shadow_prices` после решения прямой задачи.

    Исключения:
        RuntimeError: Если solver не смог найти оптимальное решение.
    """

    # TODO: задайте `c = -effects` и вызовите `linprog` для прямой задачи.
    return None, None


def solve_dual(effects, A_ub, b_ub):
    """Собирает и решает двойственную задачу для модели с верхними границами.

    Аргументы:
        effects (np.ndarray): Вектор эффектов прямой задачи.
        A_ub (np.ndarray): Матрица ресурсных ограничений прямой задачи.
        b_ub (np.ndarray): Вектор правых частей ресурсных ограничений.

    Возвращает:
        scipy.optimize.OptimizeResult | None: Результат решения двойственной задачи.

    Исключения:
        RuntimeError: Если solver не смог найти оптимальное решение.
    """

    # TODO: соберите `c_dual`, `A_dual`, `b_dual` и границы двойственных переменных.
    return None


def rerun_with_resource_change(effects, A_ub, b_ub, bounds, resource_index, delta):
    """Пересчитывает модель после изменения одного ресурсного лимита.

    Аргументы:
        effects (np.ndarray): Вектор эффектов программ.
        A_ub (np.ndarray): Матрица расхода ресурсов.
        b_ub (np.ndarray): Исходный вектор лимитов ресурсов.
        bounds (list[tuple[float, float]]): Границы переменных прямой задачи.
        resource_index (int): Индекс ресурса, который меняется в сценарии.
        delta (float): Приращение правой части выбранного ограничения.

    Возвращает:
        tuple | None: Новый вектор лимитов и результат повторного решения модели.

    Исключения:
        RuntimeError: Если solver не смог найти оптимальное решение.
    """

    # TODO: скопируйте `b_ub`, измените один ресурс и снова решите прямую задачу.
    return None


# TODO: подготовьте задачу максимизации через минимизацию отрицательной цели.
c = None
result = None
dual_result = None
shadow_prices = None
slack = None
binding = None

# TODO: после своей попытки решите модель и заполните анализ.
# c = -effects
# result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
# shadow_prices = -result.ineqlin.marginals
# slack = result.slack
# binding = np.isclose(slack, 0.0)
# dual_result = solve_dual(effects, A_ub, b_ub)
# np.allclose(-result.fun, dual_result.fun)

## 6. Что должно быть в отчёте

1. Прямая постановка задачи.
2. Объяснение двойственной задачи как зеркала прямой.
3. Оптимальный план по программам.
4. Таблица активных ограничений и запасов.
5. Таблица теневых цен ресурсов.
6. Единица измерения каждой теневой цены.
7. Проверка сильной двойственности.
8. Объяснение нулевых shadow prices, если они есть.
9. Разделение смысла `y_i` и `z_j`.
10. Минимум два сценария по `b` и один по `c`.
11. Отметка, что прогноз по shadow price локален.

## 7. Контрольный чек-лист

- [ ] Я показал, какие ограничения стали binding.
- [ ] Я объяснил dual как зеркальный взгляд на primal.
- [ ] Я указал единицу измерения для каждой теневой цены.
- [ ] Я объяснил, что нулевая теневая цена не делает ресурс бесполезным вообще.
- [ ] Я отделил ресурсные переменные `y_i` от переменных верхних границ `z_j`.
- [ ] Я осмысленно интерпретировал shadow prices.
- [ ] Я сравнил прогноз и фактический пересчёт.
- [ ] Я сформулировал управленческий вывод по самому дефицитному ресурсу.